In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..")

In [ ]:
import json
import re
from pathlib import Path

from omegaconf import OmegaConf

from patient_simulator.data_extraction import extract_patient_profile_from_transcript
from patient_simulator.misc.utils import create_llm_instance

# Preprocess the VHA transcripts

Mirror the ACI-Bench preprocessing (`preprocess_aci_bench.ipynb`): clean the raw
transcripts into `D:`/`P:` conversations and extract a structured patient profile
per case. Outputs land in `data/vha/conversations` and
`data/vha/extracted_profiles`, ready to replicate the ACI-Bench run.

## Step 1: Clean up transcripts

Raw VHA transcripts carry a metadata header (`Date of Encounter`, `Time of
Encounter`, `AUDIO NAME`) and a metadata footer (`PATIENT LEAVES: 88:49`,
`Time is 1:25.`, `DD: @@/@@/@@`, `DT: ...`, `INFOPRO/.../JOB# ...`,
`TRANSACTION # ...`), plus `@@@`-style redaction masks, `HH:MM` timestamps and
`(DOCTOR OUT: ...)` stage directions. Speaker turns are labelled `DOCTOR <id>` /
`PATIENT <id>` (or glued `DOCTOR153:`), with companions labelled
`SECOND PERSON <id>`, `WIFE <id>`, etc.

Everything before the first speaker label is dropped, header/footer metadata,
date/time markers, masks and non-verbal cues are stripped, and every non-doctor
speaker is folded onto the patient side to keep a two-party `D:`/`P:` transcript.

In [ ]:
DOCTOR_RE = re.compile(
    r"^(?:(?:FIRST|SECOND|THIRD|FOURTH)\s+)?(?:DOCTOR|DR\.?)\s*(?P<id>[A-Za-z0-9]+)?[.:]?$"
)
OTHER_RE = re.compile(
    r"^(?:PATIENT|SECOND PERSON|THIRD PERSON|FOURTH PERSON|OTHER PERSON|PERSON|"
    r"WIFE|HUSBAND|SON|DAUGHTER|CAREGIVER|FAMILY|SPEAKER|INTERPRETER)\s*(?P<id>[A-Za-z0-9]+)?[.:]?$"
)
STAGE_RE = re.compile(
    r"^(?:DOCTOR|PATIENT|DR\.?|NURSE|(?:FIRST|SECOND|THIRD) (?:PERSON|DOCTOR))[^a-z]*\b"
    r"(?:LEAVES|OUT|ENTERS|RETURNS|OFF|IN)\b[^a-z]*$"
)
METADATA_RE = re.compile(
    r"^(?:DD\s*:|DT\s*:|INFOPRO|TRANSACTION\b|JOB\s*#|TIME\s+(?:STARTED|ENDED)\b|"
    r"TIME\s+IS\b\s*:?\s*[\d@]|PATIENT\s+LEAVES\b)",
    re.IGNORECASE,
)
MULTI_DOCTOR_RE = re.compile(r"^(?:FIRST|SECOND|THIRD|FOURTH)\s+DOCTOR\b", re.MULTILINE)
INLINE_STAGE_RE = re.compile(
    r"\b(DOCTOR|PATIENT|DR\.?|NURSE)\s+(OUT|IN|LEAVES|ENTERS|RETURNS)\b:?"
)
MASK_RE = re.compile(r"@+(?:[/:.-]@+)*")
TIME_RE = re.compile(r"\b\d{1,2}:\d{2}\b")
BRACKET_RE = re.compile(r"\([^)]*\)")
PUNCT_CLUSTER_RE = re.compile(r"[,.;:!?](?:\s*[,.;:!?])+")


def has_multiple_doctors(content: str) -> bool:
    return bool(MULTI_DOCTOR_RE.search(content))


def label_speaker(line: str, stem: str) -> str | None:
    stripped = line.strip()
    for regex, speaker in ((DOCTOR_RE, "D"), (OTHER_RE, "P")):
        match = regex.match(stripped)
        if not match:
            continue
        ident = match.group("id")
        if ident is None or any(ch.isdigit() for ch in ident) or ident == stem:
            return speaker
    return None


def _collapse_punct(match: re.Match) -> str:
    marks = re.sub(r"\s", "", match.group(0))
    return next((mark for mark in ".!?" if mark in marks), marks[0]) + " "


def tidy(text: str) -> str:
    text = BRACKET_RE.sub("", text)
    text = INLINE_STAGE_RE.sub(" ", text)
    text = TIME_RE.sub(" ", text)
    text = MASK_RE.sub("", text)
    text = text.replace("(", " ").replace(")", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = PUNCT_CLUSTER_RE.sub(_collapse_punct, text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r"^[\s,.;:!?]+", "", text)
    return text.strip()


def clean_transcript(content: str, stem: str) -> str:
    lines = content.splitlines()
    start = next((i for i, line in enumerate(lines) if label_speaker(line, stem)), None)
    if start is None:
        raise ValueError(f"No speaker labels found for {stem}")

    turns: list[tuple[str, str]] = []
    speaker, body = None, []
    for line in lines[start:]:
        current = label_speaker(line, stem)
        if current:
            if speaker:
                turns.append((speaker, " ".join(body)))
            speaker, body = current, []
            continue
        stripped = line.strip()
        if not stripped or STAGE_RE.match(stripped) or METADATA_RE.match(stripped):
            continue
        body.append(stripped)
    if speaker:
        turns.append((speaker, " ".join(body)))

    merged: list[tuple[str, str]] = []
    for speaker, text in turns:
        text = tidy(text)
        if not text:
            continue
        if merged and merged[-1][0] == speaker:
            merged[-1] = (speaker, tidy(f"{merged[-1][1]} {text}"))
        else:
            merged.append((speaker, text))

    return "\n\n".join(f"{speaker}: {text}" for speaker, text in merged) + "\n"

### Select cases

Keep every single-doctor transcript and drop any conversation with more than one
doctor (`FIRST/SECOND/THIRD DOCTOR` labels), since a second physician entering
breaks the two-party assumption.

In [ ]:
DATA_DIR = Path("../data/vha")
TRANSCRIPTS_DIR = DATA_DIR / "Transcripts"
CONVERSATIONS_DIR = DATA_DIR / "conversations"
PROFILES_DIR = DATA_DIR / "extracted_profiles"


def is_single_doctor(stem: str) -> bool:
    content = (TRANSCRIPTS_DIR / f"{stem}.txt").read_text(
        encoding="utf-8", errors="replace"
    )
    return not has_multiple_doctors(content)


all_stems = sorted(p.stem for p in TRANSCRIPTS_DIR.glob("*.txt"))
case_ids = [s for s in all_stems if is_single_doctor(s)]
print(f"{len(case_ids)} single-doctor transcripts (of {len(all_stems)} total)")
case_ids

In [ ]:
CONVERSATIONS_DIR.mkdir(parents=True, exist_ok=True)

for case_id in case_ids:
    raw = (TRANSCRIPTS_DIR / f"{case_id}.txt").read_text(
        encoding="utf-8", errors="replace"
    )
    cleaned = clean_transcript(raw, case_id)
    (CONVERSATIONS_DIR / f"{case_id}.txt").write_text(cleaned, encoding="utf-8")

print(f"Cleaned {len(case_ids)} transcripts into {CONVERSATIONS_DIR}")

## Step 2: Extract patient profiles

Extract a structured profile per case from the cleaned conversation with
`extract_patient_profile_from_transcript`, using the same Vertex model config as
the benchmark runs.

In [ ]:
model_cfg = OmegaConf.to_container(
    OmegaConf.load("../configs/experiment/models/gemini_3_1_flash_lite_preview.yaml"),
    resolve=True,
)
llm = create_llm_instance(model_cfg, keys_file="../keys.json")

In [ ]:
import asyncio

PROFILES_DIR.mkdir(parents=True, exist_ok=True)
CONCURRENCY = 8


async def extract_and_save_profiles(case_ids: list[str]) -> None:
    semaphore = asyncio.Semaphore(CONCURRENCY)

    async def worker(case_id: str) -> None:
        out_path = PROFILES_DIR / f"{case_id}.json"
        if out_path.exists():
            print(f"{case_id}: skipped (already exists)")
            return
        async with semaphore:
            profile = await extract_patient_profile_from_transcript(
                transcript_path=str(CONVERSATIONS_DIR / f"{case_id}.txt"),
                llm=llm,
            )
        out_path.write_text(json.dumps(profile, indent=4), encoding="utf-8")
        print(f"{case_id}: done")

    await asyncio.gather(*(worker(case_id) for case_id in case_ids))


await extract_and_save_profiles(case_ids)
print(f"\nProfiles saved to {PROFILES_DIR}")

In [ ]:
sample_id = case_ids[0]
print(f"Sample profile ({sample_id}):")
print((PROFILES_DIR / f"{sample_id}.json").read_text())